# 🎨 Artistic Style Transfer
**Implémentation de Gatys et al. (2015)** — *A Neural Algorithm of Artistic Style*

Ce notebook réalise le transfert de style artistique :
1. Charge une image de **contenu** et une image de **style**
2. Extrait les features VGG19
3. Optimise une image pour minimiser les pertes de contenu + style
4. Affiche et sauvegarde le résultat

In [ ]:
import sys
sys.path.insert(0, '..')   # pour importer depuis src/

import torch
import matplotlib.pyplot as plt
from PIL import Image

from src.models.style_transfer import StyleTransfer
from src.utils.image_utils import load_image, save_image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

## 1. Chargement des images

In [ ]:
CONTENT_PATH = '../data/content/your_photo.jpg'   # ← modifiez ici
STYLE_PATH   = '../data/styles/starry_night.jpg'  # ← modifiez ici
IMAGE_SIZE   = 512

content_img = load_image(CONTENT_PATH, size=IMAGE_SIZE)
style_img   = load_image(STYLE_PATH,   size=IMAGE_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(content_img); axes[0].set_title('Contenu',  fontsize=14); axes[0].axis('off')
axes[1].imshow(style_img);   axes[1].set_title('Style',    fontsize=14); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 2. Paramètres

In [ ]:
# Poids des pertes
CONTENT_WEIGHT = 1.0
STYLE_WEIGHT   = 1e6
TV_WEIGHT      = 1e-4

# Optimisation
NUM_STEPS      = 300
OPTIMIZER      = 'lbfgs'    # 'lbfgs' | 'adam'
INIT_FROM      = 'content'  # 'content' | 'style' | 'noise'

## 3. Style Transfer

In [ ]:
loss_history = {'total': [], 'content': [], 'style': [], 'tv': []}

def on_progress(step, losses):
    for k, v in losses.items():
        loss_history[k].append(v)
    if step % 50 == 0 or step == 1:
        print(f'Step {step:4d} | total={losses["total"]:.4f} '
              f'| content={losses["content"]:.4f} '
              f'| style={losses["style"]:.6f}')

model = StyleTransfer(
    device=DEVICE,
    image_size=IMAGE_SIZE,
    content_weight=CONTENT_WEIGHT,
    style_weight=STYLE_WEIGHT,
    tv_weight=TV_WEIGHT,
)

result = model.transfer(
    content_path=CONTENT_PATH,
    style_path=STYLE_PATH,
    num_steps=NUM_STEPS,
    optimizer_type=OPTIMIZER,
    init_from=INIT_FROM,
    progress_callback=on_progress,
)

print('\n✅ Terminé !')

## 4. Résultat & Courbes de perte

In [ ]:
# --- Affichage ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(content_img); axes[0].set_title('Contenu',  fontsize=14); axes[0].axis('off')
axes[1].imshow(style_img);   axes[1].set_title('Style',    fontsize=14); axes[1].axis('off')
axes[2].imshow(result);      axes[2].set_title('Résultat', fontsize=14); axes[2].axis('off')
plt.suptitle('Artistic Style Transfer', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../output/comparison.jpg', dpi=150, bbox_inches='tight')
plt.show()

# --- Courbes ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(loss_history['content']); axes[0].set_title('Content Loss'); axes[0].set_xlabel('Step')
axes[1].plot(loss_history['style'],  color='orange'); axes[1].set_title('Style Loss'); axes[1].set_xlabel('Step')
axes[2].plot(loss_history['total'],  color='green');  axes[2].set_title('Total Loss'); axes[2].set_xlabel('Step')
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Sauvegarde

In [ ]:
OUTPUT_PATH = '../output/result.jpg'
save_image(result, OUTPUT_PATH)
print(f'Image sauvegardée : {OUTPUT_PATH}')